# Experiment 17 — ADI as a predictor of downstream $\Delta$AUC

The Artifact Detectability Index (ADI, geometric mean of per-artifact
linear-probe CLS AUCs on the standardized battery $\mathcal{B}_\text{CXR}$)
is validated against the mean absolute downstream $\Delta$AUC from exp04.
Per-(dataset, model) ADI values are correlated with per-(dataset, model)
mean $\vert\Delta$AUC$\vert$ using Spearman rank correlation; 95% CI is
obtained by 2000-iteration bootstrap over (dataset, model) pairs. A strong
positive correlation evidences that ADI tracks downstream disease
classification perturbation magnitude.


In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path(os.environ.get('V4_WORK_DIR',
    '/home/saptpurk/embeddings-noise-eliminators/v4_work'))
out_dir = ROOT / 'v4_exp17_adi_vs_delta'
out_dir.mkdir(parents=True, exist_ok=True)

def _gmean(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return float(np.exp(np.log(x).mean())) if len(x) else np.nan

artifact_dfs = []
for pat, exp in [('v4_exp02_geometric_*/*.parquet', 'exp02'),
                 ('v4_exp03_iso_motion_blur_*/*.parquet', 'exp03'),
                 ('v4_exp06_patch_probing_*/*.parquet', 'exp06'),
                 ('v4_exp08_directional_blur_*/*.parquet', 'exp08')]:
    for f in sorted(ROOT.glob(pat)):
        d = pd.read_parquet(f); d['exp'] = exp
        if 'dataset' not in d.columns:
            d['dataset'] = f.parent.name.split('_')[-1]
        if exp == 'exp06':
            d = d[d.get('pooling', 'cls') == 'cls']
        artifact_dfs.append(d)

if not artifact_dfs:
    pd.DataFrame({'status': ['inputs_absent']}).to_parquet(
        out_dir / 'exp17_manifest.parquet', index=False)
else:
    arts = pd.concat(artifact_dfs, ignore_index=True)
    adi = (arts.groupby(['dataset', 'model'])['auc']
                .apply(_gmean).reset_index(name='adi'))

    ds = []
    for f in sorted(ROOT.glob('v4_exp04_clean_vs_perturbed_*/*.parquet')):
        ds.append(pd.read_parquet(f))
    if not ds:
        pd.DataFrame({'status': ['exp04_absent']}).to_parquet(
            out_dir / 'exp17_manifest.parquet', index=False)
    else:
        e4 = pd.concat(ds, ignore_index=True)
        delta = (e4.assign(abs_delta=e4['delta_auc'].abs())
                   .groupby(['dataset', 'model'])['abs_delta']
                   .mean().reset_index(name='mean_abs_delta_auc'))
        merged = adi.merge(delta, on=['dataset', 'model'], how='inner')

        if len(merged) >= 3:
            rho, p = stats.spearmanr(merged['adi'], merged['mean_abs_delta_auc'])
            rng = np.random.default_rng(42); boots = []
            for _ in range(2000):
                idx = rng.choice(len(merged), len(merged), replace=True)
                if merged.iloc[idx]['adi'].nunique() < 2: continue
                boots.append(stats.spearmanr(merged.iloc[idx]['adi'],
                    merged.iloc[idx]['mean_abs_delta_auc']).correlation)
            lo, hi = np.percentile(boots, [2.5, 97.5])
            summary = pd.DataFrame([{'spearman_rho': rho, 'spearman_p': p,
                'ci_low': float(lo), 'ci_high': float(hi),
                'n_points': len(merged)}])
        else:
            summary = pd.DataFrame([{'spearman_rho': np.nan,
                'spearman_p': np.nan, 'ci_low': np.nan, 'ci_high': np.nan,
                'n_points': len(merged)}])

        merged.to_parquet(out_dir / 'exp17_adi_vs_delta_points.parquet', index=False)
        summary.to_parquet(out_dir / 'exp17_adi_vs_delta_summary.parquet', index=False)
        print(summary.to_string(index=False))
